# 05 - Capstone: Training and Exporting the Detector (Python)

This is step 1 of the three-language pipeline described in `concept.md`: train a small classifier that decides whether a detected camera blob is a real game piece or noise, then export it to ONNX so `cpp/infer.cpp` can load and run it without any Python installed at all.

Run this notebook top to bottom before building `cpp/infer.cpp` — it produces the `cpp/detector.onnx` file that step depends on.

## The Feature Vector

We're deliberately not working with raw images here — a simpler upstream vision step (color thresholding, contour detection) has already isolated a candidate blob in the camera frame, and reduced it to four numbers:

- `width` — the blob's bounding box width, normalized to roughly [0, 1] as a fraction of the frame
- `height` — same, for height
- `aspect_ratio` — `width / height`
- `fill_ratio` — the fraction of the bounding box the blob's actual pixels fill in

Real game pieces tend to be roughly round or square (aspect ratio near 1) and solidly filled (high fill ratio). Noise — shadows, reflections, stray carpet — tends to be more irregular in both respects. That's the pattern the model needs to learn.

## Imports

In [1]:
import numpy as np
import torch
import torch.nn as nn
import onnxruntime as ort

np.random.seed(1515)   # team number, for reproducibility -- see ml_resources for the same convention
torch.manual_seed(1515)

## Generating a Synthetic Dataset

We don't have a labeled dataset of real camera detections handy, so we simulate one: draw `game piece` examples from one distribution and `noise` examples from another, with enough overlap that the classification problem isn't trivially easy.

In [2]:
def generate_dataset(n_samples):
    labels = np.random.randint(0, 2, n_samples)
    features = np.zeros((n_samples, 4), dtype=np.float32)

    # label 1: game piece -- roughly square/round, solidly filled
    piece_mask = labels == 1
    n_pieces = piece_mask.sum()
    features[piece_mask, 0] = np.random.normal(0.18, 0.03, n_pieces)   # width
    features[piece_mask, 1] = np.random.normal(0.18, 0.03, n_pieces)   # height
    features[piece_mask, 2] = np.random.normal(1.00, 0.08, n_pieces)   # aspect_ratio
    features[piece_mask, 3] = np.random.normal(0.75, 0.07, n_pieces)   # fill_ratio

    # label 0: noise -- more irregular shape, less solidly filled
    noise_mask = labels == 0
    n_noise = noise_mask.sum()
    features[noise_mask, 0] = np.random.normal(0.12, 0.05, n_noise)
    features[noise_mask, 1] = np.random.normal(0.20, 0.06, n_noise)
    features[noise_mask, 2] = np.random.normal(1.40, 0.30, n_noise)
    features[noise_mask, 3] = np.random.normal(0.35, 0.12, n_noise)

    features[:, 0:2] = np.clip(features[:, 0:2], 0.01, 1.0)
    features[:, 2] = np.clip(features[:, 2], 0.1, 5.0)
    features[:, 3] = np.clip(features[:, 3], 0.0, 1.0)

    return features, labels.astype(np.int64)


X_all, y_all = generate_dataset(2000)

split = int(0.8 * len(X_all))
X_train, y_train = X_all[:split], y_all[:split]
X_test, y_test = X_all[split:], y_all[split:]

print("train examples:", len(X_train), " test examples:", len(X_test))
print("example feature vector:", X_train[0], " label:", y_train[0])

train examples: 1600  test examples: 400
example feature vector: [0.2027487  0.18650998 1.0518402  0.7783159 ]  label: 1


## Defining and Training a Small Model

Four inputs, one small hidden layer, two output scores (one per class: "noise" and "game piece"). This is intentionally tiny — the point of this capstone is the pipeline connecting the three languages, not squeezing out maximum accuracy from a large model.

In [3]:
class DetectorNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, x):
        return self.net(x)  # raw logits -- softmax is applied later, not inside the model


model = DetectorNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)

for epoch in range(200):
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0 or epoch == 199:
        print(f"epoch {epoch:>3}: loss = {loss.item():.4f}")

epoch   0: loss = 0.7623
epoch  50: loss = 0.4021
epoch 100: loss = 0.1448
epoch 150: loss = 0.0816
epoch 199: loss = 0.0589


## Evaluating on Held-Out Data

In [4]:
model.eval()
with torch.no_grad():
    X_test_t = torch.from_numpy(X_test)
    test_logits = model(X_test_t)
    test_predictions = test_logits.argmax(dim=1).numpy()

accuracy = (test_predictions == y_test).mean()
print(f"test accuracy: {accuracy:.2f}")

test accuracy: 0.99


## Exporting to ONNX

This is the handoff point. `torch.onnx.export` traces the model and writes out its structure and learned weights as a `.onnx` file — a format `cpp/infer.cpp` can load with no PyTorch, and no Python, installed anywhere on the machine that runs it. We fix the input to a single feature vector (`shape (1, 4)`) since the C++ side always classifies one detection at a time, matching how a real per-frame inference call would work.

You'll see a block of informational `[torch.onnx] ...` progress logging below (and a `torchvision is not installed` warning, since we aren't using torchvision anywhere) — that's expected and not an error.

In [5]:
import os

dummy_input = torch.zeros(1, 4, dtype=torch.float32)

torch.onnx.export(
    model,
    dummy_input,
    "cpp/detector.onnx",
    input_names=["features"],
    output_names=["logits"],
)

# The exporter sometimes writes an unused, empty "external data" companion
# file alongside small models like this one -- harmless if left in place,
# but not needed, so we clean it up.
stray_data_file = "cpp/detector.onnx.data"
if os.path.exists(stray_data_file) and os.path.getsize(stray_data_file) == 0:
    os.remove(stray_data_file)

print("exported cpp/detector.onnx")

W0704 15:02:21.727000 35267 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms


W0704 15:02:21.727000 35267 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_align


W0704 15:02:21.728000 35267 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_pool


W0704 15:02:21.728000 35267 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::deform_conv2d


[torch.onnx] Obtain model graph for `DetectorNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DetectorNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
exported cpp/detector.onnx


/Users/nickmelamed/miniforge3/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


## Sanity-Checking the Export

Before trusting `cpp/infer.cpp` to load this file correctly, we load it back ourselves with `onnxruntime` (the same library, via its Python bindings) and confirm it produces the same predictions as the original PyTorch model on a few examples. If these don't match, something went wrong in the export step, and it's much easier to catch that here than after writing a page of C++.

In [6]:
session = ort.InferenceSession("cpp/detector.onnx")

# The export fixed the batch size at 1 (matching how cpp/infer.cpp will call
# it -- one detection at a time), so we feed it one row at a time here too.
sample_inputs = X_test[:5]
onnx_predictions = []
for row in sample_inputs:
    onnx_logits = session.run(["logits"], {"features": row.reshape(1, 4)})[0]
    onnx_predictions.append(int(onnx_logits.argmax(axis=1)[0]))

with torch.no_grad():
    pytorch_predictions = model(torch.from_numpy(sample_inputs)).argmax(dim=1).numpy()

print("onnxruntime predictions:", onnx_predictions)
print("pytorch predictions:    ", list(pytorch_predictions))
print("match:", list(onnx_predictions) == list(pytorch_predictions))

onnxruntime predictions: [1, 0, 0, 1, 1]
pytorch predictions:     [1, 0, 0, 1, 1]
match: True


## Try It Yourself

No solutions are provided — these are meant to be worked through on your own or with a mentor.

1. Add a third class — `"ambiguous"` — to `generate_dataset`, retrain with `nn.Linear(8, 3)` as the final layer, and update the export/sanity-check cells accordingly.
2. Increase the overlap between the two distributions in `generate_dataset` (move their means closer together) and see how much test accuracy drops.
3. Print the model's softmax *probabilities* (not just the predicted class) for a few test examples using `torch.softmax(test_logits, dim=1)`, and compare high-confidence correct predictions against low-confidence ones near the decision boundary.

In [7]:
# Your code here


## Resources

- [PyTorch: `torch.onnx.export`](https://pytorch.org/docs/stable/onnx.html) - the official export API used above.
- [ONNX Runtime Python API](https://onnxruntime.ai/docs/api/python/) - what we used to sanity-check the export.
- [ONNX format overview](https://onnx.ai/onnx/intro/) - what's actually inside a `.onnx` file.